In [1]:
# get a fixed read_noise, fix and forget'bout it.

import os
from pathlib import Path
import jwst
print(jwst.__version__)
from jwst import datamodels
from jwst.datamodels import dqflags

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from scipy.interpolate import interp1d
from scipy.optimize import brentq

from scipy.optimize import curve_fit, minimize
from scipy.stats import norm, poisson
from scipy.special import gamma

from pathlib import Path

# import natural units:
import natural_units as nu
# multi-core/thread:
import concurrent.futures

1.15.1


In [2]:
# 全局变量 一波定义完
dn_min = -200
dn_max = 400
dn_range = range(dn_min, dn_max)
# define the mass and cross section grid we are working with.
log_m_min  = -3
log_m_max  = 1
n_m    = 17   # From 1e-3  to 10 GeV
#log cs shift from the balloon line
cs_logshift_min = -6
cs_logshift_max = -3
cs_shift_numbers   = 25
m_grid      = np.logspace(log_m_min, log_m_max, n_m)
cs_logshift = np.linspace(cs_logshift_min, cs_logshift_max, cs_shift_numbers)
center_line = np.array([2.15504637e-23, 1.53030461e-23, 1.26359147e-23, 1.61669130e-23,
       2.40008514e-23, 4.03532201e-23, 6.95747264e-23, 1.40957345e-22,
       2.84518575e-22, 5.17381239e-22, 1.08351297e-21, 2.15203017e-21,
       4.12016417e-21, 7.78952222e-21, 1.45615810e-20, 2.70914228e-20,
       4.94000000e-20])

chi2_thr = 2.69  # For 95% CL

signal_grid = np.loadtxt('../SHIELDING_RESULT/signal_grid.txt')  # horizontal mass, vertical cs

output_dir = './advan_constraint/frac_1/'
result_dir = output_dir + 'constraints/'
delta_chi2_dir = output_dir + 'delta_chi2/'
if not os.path.exists(result_dir):
    os.makedirs(result_dir)
if not os.path.exists(delta_chi2_dir):
    os.makedirs(delta_chi2_dir)
likelihood_output_dir = output_dir + 'log_likelihood_grid/'

In [3]:
def generate_constraint_grid(filename):
    input_file_base = filename
    #naive_constraint = np.loadtxt('./naive_constraint/' + input_file_base +'.txt')
    if os.path.exists(likelihood_output_dir + input_file_base +'_log_likelihood_grid.txt'):
        chi2_grid = -2 * np.loadtxt(likelihood_output_dir + input_file_base +'_log_likelihood_grid.txt',skiprows=1)
        result_bkd = -2 * np.loadtxt(likelihood_output_dir + input_file_base +'_log_likelihood_grid.txt', max_rows=1)
        constraint_grid = np.zeros((cs_shift_numbers, n_m))
        delta_chi2_grid = np.zeros((cs_shift_numbers, n_m))
        for i in range(n_m):
            if result_bkd <= np.min(chi2_grid[i,:]):
                chi2_min = result_bkd
                signal_min = 0
            else:
                chi2_min = np.min(chi2_grid[i,:])
                min_index = np.argmin(chi2_grid[i,:])
                signal_min = signal_grid[cs_shift_numbers-1-min_index,i]
            test_chi2 = chi2_grid[i,:] - chi2_min - chi2_thr
            for j in range(cs_shift_numbers):
                delta_chi2_grid[cs_shift_numbers-1-j,i] = chi2_grid[i,j] - chi2_min
                if test_chi2[j]>0 and signal_grid[cs_shift_numbers-1-j,i] >= signal_min:
                    constraint_grid[cs_shift_numbers-1-j,i] = 1    #cs从高到低排列
        np.savetxt(result_dir + input_file_base +'_constraint_grid.txt', constraint_grid, fmt='%d')
        np.savetxt(delta_chi2_dir + input_file_base +'_delta_chi2_grid.txt', delta_chi2_grid, fmt='%8.3f')

In [4]:
file_list = ['jw01121130001_02102_00001_nrs2'] #, 'jw01121124001_02102_00001_nrs2' ,'jw01121120001_02102_00001_nrs2']
for file in file_list:
    generate_constraint_grid(file)